# 04 — Data Visualization for LLM Benchmarks

Create publication-quality charts comparing model performance.

**Concepts:** Bar charts, line plots, throughput charts, matplotlib customization, chart selection rationale

Reuses benchmark data from notebook 03.

Requires Ollama running with all 4 benchmark models pulled.

---

## Definition

Data visualization turns benchmark metrics (latency, throughput) into interpretable charts. A bar chart comparing 4 models at 3 prompt lengths is far more informative than a table of 12 numbers.

This notebook uses `matplotlib` directly and also the `BenchmarkVisualizer` class from `src/visualization.py` to produce:

1. **Latency bar chart** — cross-model comparison at each prompt length
2. **Latency line chart** — scaling behavior as prompt length increases
3. **Throughput bar chart** — tokens per second comparison
4. **Radar chart** (with normalization) — multi-metric summary

## Motivation

Numbers in tables are hard to parse. Charts reveal patterns instantly:

- Which model dominates at all prompt lengths?
- Is there a crossover point where model B beats model A?
- How linear is the latency scaling?

Charts also communicate findings to stakeholders more effectively than raw tables. They are essential for reports, presentations, and portfolio projects.

## Theory: Chart Selection

**Which chart type for which question:**

| Chart Type | Question It Answers | Used For |
|---|---|---|
| Grouped bar chart | "Which model is fastest at X prompt length?" | Cross-model comparison at fixed points |
| Line chart | "How does latency grow with prompt length?" | Scaling behavior, trend detection |
| Radar chart (normalized) | "What is each model's overall profile?" | Multi-metric summary (requires normalization) |

**Critical principle: radar charts require normalization** for lower-is-better metrics. Plotting raw latency on a radar causes higher-latency (worse) models to appear larger/better. Normalize via:

```
normalized = 1 - (value - min) / (range)  → higher = better
```

or use a positively-oriented metric like throughput (tok/s).

## Real-World Examples

| Audience | Chart Needed | Insight |
|---|---|---|
| Engineering team | Latency bar chart | "Qwen3.5:2b is 3x faster than 4b version" |
| Product manager | Throughput chart | "We can handle 50 concurrent requests with Granite" |
| Research paper | Radar chart | "Model X achieves best balance across all metrics" |
| Blog post | Line chart + trendline | "Latency scales O(n^1.1) with prompt length" |

## Visual Explanation

**Charts generated in this notebook:**

```
data → BenchmarkRunner → dict → generate_all()
                                    │
                    ┌───────────────┼───────────────┐
                    ▼               ▼               ▼
              latency_bar     latency_line      throughput
              (grouped bar)    (line plot)      (grouped bar)
                    │               │               │
                    └───────────────┼───────────────┘
                                    ▼
                              radar.png
                         (normalized radar)
```

Each chart is saved to `outputs/figures/` at 150 DPI for publication quality.

## Code Walkthrough

In [ ]:
# 1. Setup: import libraries, create output directory
import matplotlib
matplotlib.use("Agg")  # headless backend — no display required
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

FIGS = Path("outputs/figures")
FIGS.mkdir(parents=True, exist_ok=True)
print(f"Figures directory: {FIGS.resolve()}")

In [ ]:
# 2. Load benchmark data
# Re-runs benchmark fresh (in production, save/load from disk)
from src.benchmarking import BenchmarkRunner

runner = BenchmarkRunner()
data = runner.run_all()
runner.close()

models = list(data.keys())
labels = list(next(iter(data.values())).keys())
print(f"Data loaded: {len(models)} models x {len(labels)} lengths")

In [ ]:
# 3. Bar chart: latency comparison
# Grouped bars: each model gets one bar per prompt length
# Annotation on each bar shows exact value
x = np.arange(len(labels))
w = 0.2  # bar width
fig, ax = plt.subplots(figsize=(10, 6))
for i, m in enumerate(models):
    vals = [data[m][l]["latency_s"] for l in labels]
    bars = ax.bar(x + i * w, vals, w, label=m)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f"{v:.1f}s", ha="center", va="bottom", fontsize=8)
ax.set_xlabel("Prompt Length")
ax.set_ylabel("Latency (s)")
ax.set_title("Inference Latency by Model")
ax.set_xticks(x + w * (len(models) - 1) / 2)
ax.set_xticklabels(labels)
ax.legend()
fig.tight_layout()
fig.savefig(FIGS / "latency_bar.png", dpi=150)
plt.close(fig)
print("latency_bar.png saved")

In [ ]:
# 4. Line chart: latency scaling
# Shows how each model's latency grows with prompt length
# Linear scale suggests predictable scaling; super-linear indicates memory pressure
fig, ax = plt.subplots(figsize=(10, 6))
for m in models:
    vals = [data[m][l]["latency_s"] for l in labels]
    ax.plot(labels, vals, marker="o", label=m, linewidth=2)
ax.set_xlabel("Prompt Length")
ax.set_ylabel("Latency (s)")
ax.set_title("Latency Scaling with Prompt Length")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGS / "latency_line.png", dpi=150)
plt.close(fig)
print("latency_line.png saved")

In [ ]:
# 5. Throughput bar chart
# tokens/second is a positively-oriented metric (higher = better)
# Better for comparing "how much work can each model do per second"
fig, ax = plt.subplots(figsize=(10, 6))
for i, m in enumerate(models):
    vals = [data[m][l]["tokens"] / max(data[m][l]["latency_s"], 0.001) for l in labels]
    bars = ax.bar(x + i * w, vals, w, label=m)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{v:.0f}", ha="center", va="bottom", fontsize=8)
ax.set_xlabel("Prompt Length")
ax.set_ylabel("Tokens / Second")
ax.set_title("Throughput by Model")
ax.set_xticks(x + w * (len(models) - 1) / 2)
ax.set_xticklabels(labels)
ax.legend()
fig.tight_layout()
fig.savefig(FIGS / "throughput.png", dpi=150)
plt.close(fig)
print("throughput.png saved")

In [ ]:
# 6. Radar chart (normalized)
# Plotting raw latency on radar is misleading (lower is better, but radar shows area)
# We normalize: normalized = 1 - (value - min)/(range) so higher = better for all axes
# Then plot throughput (tok/s) directly since it's positively-oriented
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"projection": "polar"})
for m in models:
    vals = [data[m][l]["tokens"] / max(data[m][l]["latency_s"], 0.001) for l in labels]
    vals += vals[:1]
    ax.plot(angles, vals, label=m)
    ax.fill(angles, vals, alpha=0.05)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels)
ax.set_title("Model Throughput Radar (tok/s)")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(FIGS / "radar.png", dpi=150)
plt.close(fig)
print("radar.png saved")

In [ ]:
# 7. Display all generated charts
from IPython.display import Image, display
for p in sorted(FIGS.glob("*.png")):
    print(f"\n--- {p.name} ---")
    display(Image(filename=str(p)))

## Best Practices

1. **Use `fig.tight_layout()`** — prevents axis labels and titles from being clipped
2. **Save at 150+ DPI** — ensures chart is publication-ready for reports and presentations
3. **Close figures with `plt.close(fig)`** — prevents memory leaks in notebooks
4. **Label axes and add legends** — charts are useless without context
5. **Normalize radar charts** — plotting raw lower-is-better metrics on radar is misleading
6. **Use consistent color schemes** — map each model to the same color across all charts

## Common Mistakes

| Mistake | Fix |
|---|---|
| Radar chart with raw latency | Higher latency (worse) appears as larger area — normalize or use throughput |
| No legend or unlabeled axes | Every chart must be self-explanatory |
| Cluttered bar annotations | Use `fontsize=8` and offset text above bars |
| Not closing figures | `plt.close(fig)` after each save to avoid memory growth |
| Low DPI output | 150 DPI minimum for print; 300 for publication |
| Overlapping data labels | Rotate x-axis labels or adjust figure width |

## Summary

- Bar charts best for cross-model comparison at same prompt length
- Line charts show scaling behavior — look for linear vs super-linear trends
- Throughput (tok/s) reveals practical speed for batch/inference serving
- Radar charts useful for comparing multiple metrics simultaneously — always normalize lower-is-better metrics
- Always save high-DPI (150+) PNGs for reports and presentations
- Use `fig.tight_layout()` to prevent label clipping